## Setup

In [51]:
from pathlib import Path

import pandas as pd
from torch import from_numpy, Tensor
from torchmetrics import MetricCollection
import torchmetrics


In [10]:
base = Path(".").resolve() / "eval-all" / "data"
assert base.is_dir() and base.exists()
base


PosixPath('/workspaces/ts-qa/eval-all/data')

In [17]:
results = pd.read_hdf(base / "all_joined.h5", "data")
results

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text,answer,options,options_sequence,prediction_parsed_binary,prediction_parsed_multi
0,4.2.1,Falcon (7B),100,24000,0,2,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,interval_whole_sequence_multi,What is the total count of diverse actions per...,multi,The correct answer is: 4.,2.0,"{'A': '1', 'B': '2', 'C': '4'}","[1, 2, 4]",NaN,B
1,4.2.1,Falcon (7B),100,24000,1,holding a baby,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,count_number_multi,Can you determine the activity that the person...,multi,The correct answer is holding a baby.,0.0,"{'A': 'holding a baby', 'B': 'skipping rope', ...","[holding a baby, skipping rope, eating with th...",NaN,A
2,4.2.1,Falcon (7B),100,24000,2,playing guitar,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,before_multi,What actions were undertaken by the person som...,multi,The correct answer is playing guitar.,2.0,"{'A': 'throwing a ball', 'B': 'T-posing', 'C':...","[throwing a ball, T-posing, playing guitar]",NaN,C
3,4.2.1,Falcon (7B),100,24000,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,comparison_timestamp_same_binary,Is there no notable distinction between the ac...,binary,No,0.0,NaN,NaN,True,NaN
4,4.2.1,Falcon (7B),100,24000,4,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,count_open,What is the total number of times the person p...,open,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2684995,4.2.2,UMT5 Small,full,26999,0,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","As the first action, the person is skipping ro...",comparison_counting_binary,Does the person engage in eating with the righ...,binary,Correct,1.0,NaN,NaN,False,NaN
2684996,4.2.2,UMT5 Small,full,26999,1,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","As the first action, the person is skipping ro...",comparison_first_last_same_binary,Is the initial action equivalent to the final ...,binary,No,0.0,NaN,NaN,False,NaN
2684997,4.2.2,UMT5 Small,full,26999,2,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","As the first action, the person is skipping ro...",right_after_binary,"Right after T-posing, does the person proceed ...",binary,True,1.0,NaN,NaN,True,NaN
2684998,4.2.2,UMT5 Small,full,26999,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","As the first action, the person is skipping ro...",right_before_binary,"At a certain point in time, does skipping rope...",binary,This is true!,1.0,NaN,NaN,True,NaN


## Binary Tasks

In [ ]:
binary = results[results["answer_type"] == "binary"].copy()
binary.drop(
    columns=["options", "options_sequence", "prediction_parsed_multi"], inplace=True
)
binary = binary.astype({"answer": int, "prediction_parsed_binary": float}, copy=False)
binary

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text,answer,prediction_parsed_binary
3,4.2.1,Falcon (7B),100,24000,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,comparison_timestamp_same_binary,Is there no notable distinction between the ac...,binary,No,0.0,1.0
6,4.2.1,Falcon (7B),100,24001,1,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,first_binary,Was drinking with the left hand the first acti...,binary,This is not correct.,0.0,0.0
8,4.2.1,Falcon (7B),100,24001,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,comparison_timestamp_same_binary,Is there no notable distinction between the ac...,binary,False,0.0,1.0
9,4.2.1,Falcon (7B),100,24001,4,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,interval_part_sequence_binary,Is it accurate to state that the person is eng...,binary,False,0.0,1.0
10,4.2.1,Falcon (7B),100,24002,0,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person initiates their actions with eating...,comparison_first_last_different_binary,Are the initial and final actions not the same?,binary,Not true,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2684994,4.2.2,UMT5 Small,full,26998,4,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with skipping rope. This a...,descriptive_identification_binary,Is skipping rope the action the person is enga...,binary,This is true!,1.0,0.0
2684995,4.2.2,UMT5 Small,full,26999,0,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","As the first action, the person is skipping ro...",comparison_counting_binary,Does the person engage in eating with the righ...,binary,Correct,1.0,0.0
2684996,4.2.2,UMT5 Small,full,26999,1,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","As the first action, the person is skipping ro...",comparison_first_last_same_binary,Is the initial action equivalent to the final ...,binary,No,0.0,0.0
2684997,4.2.2,UMT5 Small,full,26999,2,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","As the first action, the person is skipping ro...",right_after_binary,"Right after T-posing, does the person proceed ...",binary,True,1.0,1.0


In [64]:
def unwrap(res: dict[str, Tensor]) -> dict[str, float]:
    return {k: v.item() for k, v in res.items()}


def compute_binary_metrics(df: pd.DataFrame) -> dict[str, float]:
    metrics = MetricCollection(
        {
            "Accuracy": torchmetrics.Accuracy(task="binary"),
            "Precision": torchmetrics.Precision(task="binary"),
            "Recall": torchmetrics.Recall(task="binary"),
            "F1": torchmetrics.F1Score(task="binary"),
        }
    )

    return unwrap(
        metrics(
            from_numpy(df["prediction_parsed_binary"].to_numpy()),
            from_numpy(df["answer"].to_numpy()),
        )
    )


all_results = (
    binary.groupby(["setting", "model", "mode"])
    .apply(compute_binary_metrics, include_groups=False)
    .apply(pd.Series)
    .reset_index()
)
all_results

,setting,model,mode,Accuracy,F1,Precision,Recall
0,4.2.1,Falcon (7B),100,0.586047,0.584581,0.613928,0.557911
1,4.2.1,Falcon (7B),150,0.556173,0.523358,0.575957,0.479562
2,4.2.1,Falcon (7B),200,0.605774,0.640958,0.610972,0.674040
3,4.2.1,Falcon (7B),250,0.548196,0.379379,0.670561,0.264516
4,4.2.1,Falcon (7B),300,0.537931,0.326397,0.682975,0.214439
...,...,...,...,...,...,...,...
174,4.2.2,MVP,full,0.683729,0.662360,0.723770,0.610557
175,4.2.2,T5 Base,full,0.644154,0.598672,0.701080,0.522369
176,4.2.2,T5 Small,full,0.617642,0.575196,0.684769,0.495853
177,4.2.2,UMT5 Base,full,0.566966,0.592866,0.567560,0.620534


## Multi Tasks

In [65]:
multi = results[results["answer_type"] == "multi"].copy()
multi.drop(columns=["prediction_parsed_binary"], inplace=True)
multi

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text,answer,options,options_sequence,prediction_parsed_multi
0,4.2.1,Falcon (7B),100,24000,0,2,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,interval_whole_sequence_multi,What is the total count of diverse actions per...,multi,The correct answer is: 4.,2.0,"{'A': '1', 'B': '2', 'C': '4'}","[1, 2, 4]",B
1,4.2.1,Falcon (7B),100,24000,1,holding a baby,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,count_number_multi,Can you determine the activity that the person...,multi,The correct answer is holding a baby.,0.0,"{'A': 'holding a baby', 'B': 'skipping rope', ...","[holding a baby, skipping rope, eating with th...",A
2,4.2.1,Falcon (7B),100,24000,2,playing guitar,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,before_multi,What actions were undertaken by the person som...,multi,The correct answer is playing guitar.,2.0,"{'A': 'throwing a ball', 'B': 'T-posing', 'C':...","[throwing a ball, T-posing, playing guitar]",C
7,4.2.1,Falcon (7B),100,24001,2,drinking with the left hand,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,last_multi,Regarding the final action undertaken by the p...,multi,The correct answer is: drinking with the left ...,0.0,"{'A': 'drinking with the left hand', 'B': 'cat...","[drinking with the left hand, catching a ball,...",A
19,4.2.1,Falcon (7B),100,24003,4,picking something up with both hands,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,first_multi,What was the action performed by the person as...,multi,The correct answer is: picking something up wi...,0.0,"{'A': 'picking something up with both hands', ...","[picking something up with both hands, catchin...",A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2684977,4.2.2,UMT5 Small,full,26995,2,holding a baby,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with holding a baby. It is...,comparison_counting_multi,"Among A: holding a baby, B: kicking a ball, or...",multi,The correct answer is holding a baby.,0.0,"{'A': 'holding a baby', 'B': 'kicking a ball',...","[holding a baby, kicking a ball, waving]",A
2684978,4.2.2,UMT5 Small,full,26995,3,3,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with holding a baby. It is...,interval_whole_sequence_multi,How many different actions does the person per...,multi,The correct answer is: 3.,0.0,"{'A': '3', 'B': '2', 'C': '1'}","[3, 2, 1]",A
2684979,4.2.2,UMT5 Small,full,26995,4,playing guitar,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with holding a baby. It is...,after_multi,"Once the person finished holding a baby, what ...",multi,The correct answer is golfing (swinging a club).,0.0,"{'A': 'golfing (swinging a club)', 'B': 'playi...","[golfing (swinging a club), playing guitar, pi...",B
2684987,4.2.2,UMT5 Small,full,26997,2,bowing,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","As the first action, the person is shaking han...",right_before_multi,"Immediately prior to catching a ball, what act...",multi,The correct answer is shaking hands.,0.0,"{'A': 'shaking hands', 'B': 'drinking with the...","[shaking hands, drinking with the left hand, b...",C


In [72]:
def compute_multi_metrics(df: pd.DataFrame) -> dict[str, float]:
    metrics = MetricCollection(
        {
            "Accuracy": torchmetrics.Accuracy(task="multiclass", num_classes=3),
            "Precision": torchmetrics.Precision(task="multiclass", num_classes=3),
            "Recall": torchmetrics.Recall(task="multiclass", num_classes=3),
            "F1": torchmetrics.F1Score(task="multiclass", num_classes=3),
        }
    )

    return unwrap(
        metrics(
            from_numpy(
                df["prediction_parsed_multi"].map({"A": 0, "B": 1, "C": 2}).to_numpy()
            ),
            from_numpy(df["answer"].to_numpy()),
        )
    )


all_results_multi = (
    multi.groupby(["setting", "model", "mode"])
    .apply(compute_multi_metrics, include_groups=False)
    .apply(pd.Series)
    .reset_index()
)
all_results_multi

,setting,model,mode,Accuracy,F1,Precision,Recall
0,4.2.1,Falcon (7B),100,0.496070,0.496070,0.496070,0.496070
1,4.2.1,Falcon (7B),150,0.488725,0.488725,0.488725,0.488725
2,4.2.1,Falcon (7B),200,0.604253,0.604253,0.604253,0.604253
3,4.2.1,Falcon (7B),250,0.685160,0.685160,0.685160,0.685160
4,4.2.1,Falcon (7B),300,0.526815,0.526815,0.526815,0.526815
...,...,...,...,...,...,...,...
174,4.2.2,MVP,full,0.416705,0.416705,0.416705,0.416705
175,4.2.2,T5 Base,full,0.391164,0.391164,0.391164,0.391164
176,4.2.2,T5 Small,full,0.363847,0.363847,0.363847,0.363847
177,4.2.2,UMT5 Base,full,0.376898,0.376898,0.376898,0.376898
